# Pydantic AI Hosted Agent Setup

This notebook helps you build, deploy, and test a **Pydantic AI-based custom agent** on Microsoft Foundry Hosted Agents.

### What this notebook does

1. **Build & Push** - Builds a Docker container image for your Pydantic AI agent and pushes it to Azure Container Registry (ACR)
2. **Deploy** - Creates a hosted agent version in your Foundry project, pulling the image from ACR
3. **Test Direct** - Validates the agent works by calling Foundry's Responses API directly with your Azure credentials
4. **Test via APIM** - Routes requests through Azure API Management to test the production gateway path

### Prerequisites

Before running this notebook:
- Complete the **infrastructure deployment** notebook first (creates Foundry resources, APIM, ACR)
- Have the deployment outputs available:
  - Foundry project endpoint
  - ACR login server
  - APIM suffix
  - APIM subscription key
- Docker Desktop must be running
- You must be logged in to Azure CLI (`az login`)

## 1. Build And Push The Container Image

This builds the image locally with Docker and pushes it to ACR.
Run this after completing the prerequisites above.

In [ ]:
# Central configuration (edit here once, then run notebook top-to-bottom)
ACR_NAME = "X"
APIM_SUFFIX = "X"
APIM_SUBSCRIPTION_KEY = "X"
PROJECT_ENDPOINT = f"https://foundry-agents-{APIM_SUFFIX}.services.ai.azure.com/api/projects/default-foundry-agents"

AGENT_NAME = "pydantic-agent"
FRAMEWORK_DIR = "pydantic"
IMAGE_NAME = "pydantic-agent"
IMAGE_TAG = "1.0.0"

AZURE_OPENAI_API_VERSION = "2024-05-01-preview"
AZURE_OPENAI_DEPLOYMENT = "gpt-5-mini"
API_VERSION = "v1"
DEFAULT_CONTENT_TYPE = "application/json"
FOUNDRY_FEATURES_HEADER = "HostedAgents=V1Preview"
DIRECT_TEST_INPUT = "Hello! What can you help me with?"

ACR_LOGIN_SERVER = f"{ACR_NAME}.azurecr.io"
IMAGE_URI = f"{ACR_LOGIN_SERVER}/{IMAGE_NAME}:{IMAGE_TAG}"
AZURE_OPENAI_ENDPOINT = f"https://apim-{APIM_SUFFIX}.azure-api.net/inference/models"

def build_foundry_responses_url(project_endpoint: str, agent_name: str) -> str:
    return f"{project_endpoint}/agents/{agent_name}/endpoint/protocols/openai/responses"

def build_apim_responses_url(apim_suffix: str, agent_name: str) -> str:
    return f"https://apim-{apim_suffix}.azure-api.net/hosted-agent-responses/agents/{agent_name}/endpoint/protocols/openai/responses"

print("Configuration loaded.")
print(f"ACR_LOGIN_SERVER={ACR_LOGIN_SERVER}")
print(f"IMAGE_URI={IMAGE_URI}")
print(f"PROJECT_ENDPOINT={PROJECT_ENDPOINT}")
print(f"AZURE_OPENAI_ENDPOINT={AZURE_OPENAI_ENDPOINT}")
print(f"AGENT_NAME={AGENT_NAME}")

### Fill in your configuration

Replace the placeholder values above with your actual Azure resources. These should match the outputs from the infrastructure deployment notebook:
- `ACR_NAME`: Your Azure Container Registry name (e.g., `acrxhep4wovroziu`)
- `IMAGE_NAME`: Container image name for your Pydantic agent (usually `pydantic-agent`)
- `IMAGE_TAG`: Semantic version tag for your image (e.g., `1.0.0`)

The computed values (`ACR_LOGIN_SERVER`, `IMAGE_URI`) will be used in subsequent steps.

In [ ]:
# Ensure Docker is running
!az acr login --name {ACR_NAME}

In [ ]:
import os
import subprocess

# Build and push from the framework directory configured above
original_dir = os.getcwd()
os.chdir(FRAMEWORK_DIR)

try:
    print(f"Running: docker build -t {IMAGE_URI} .")
    subprocess.run(["docker", "build", "-t", IMAGE_URI, "--platform", "linux/amd64", "."], check=True)

    print(f"Running: docker push {IMAGE_URI}")
    subprocess.run(["docker", "push", IMAGE_URI], check=True)

    print("Docker build and push completed successfully.")
finally:
    os.chdir(original_dir)

### Build and push your container

This builds the Pydantic AI agent Docker image and pushes it to your ACR. The `--platform linux/amd64` flag ensures the image is compatible with Foundry's Linux-based container hosting.

After pushing, your image will be available at `{ACR_LOGIN_SERVER}/pydantic-agent:1.0.0` for Foundry to pull and deploy.

## 2. Create Hosted Agent Version

This section creates your Pydantic AI agent as a **Foundry Hosted Agent** using the Responses protocol v1.0.0.

### Important notes

- **Agent name**: Choose any name you want (e.g., `pydantic-agent`, `my-agent`, etc.). 
  - Foundry auto-assigns the version suffix (`:1`, `:2`, etc.) when you create versions
  - This name will be used in the APIM URL path to invoke the agent: `/agents/{agentName}/endpoint/protocols/openai/responses`

- **Model endpoint**: Uses APIM **inference** API for model calls (`https://apim-{suffix}.azure-api.net/inference/models`), not the hosted-agent API.

- **Dependencies**: Install `azure-ai-projects` and `azure-identity` to interact with Foundry APIs.

After creation, your agent will transition through status changes. Once it reaches "Running", it's ready to accept requests in sections 3 and 4.

In [ ]:
pip install azure-ai-projects==2.3.0 

### Create the hosted agent in Foundry

This cell deploys your Pydantic AI container as a **Hosted Agent** in your Foundry project. 

**Key configuration:**
- **Protocol**: Responses v1.0.0 (standard for Foundry hosted agents)
- **Resources**: 1 CPU, 2Gi memory (adjust based on your agent needs)
- **Environment variables**: 
  - `AZURE_OPENAI_ENDPOINT`: Points to APIM inference API for your agent to call models
  - `APIM_SUBSCRIPTION_KEY`: APIM subscription key for authentication
  - `AZURE_OPENAI_DEPLOYMENT`: Model name to use (e.g., `gpt-5-mini`)

The agent will be created with a version suffix (auto-assigned by Foundry). After creation, it transitions to "Running" state and is ready to accept requests via the Responses API.

In [ ]:
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import (
    HostedAgentDefinition,
    ProtocolVersionRecord,
    AgentEndpointProtocol,
    ContainerConfiguration,
 )
from azure.identity import AzureCliCredential

print(f"PROJECT_ENDPOINT: {PROJECT_ENDPOINT}")
print(f"AGENT_NAME: {AGENT_NAME}")
print(f"IMAGE_URI: {IMAGE_URI}")
print(f"AZURE_OPENAI_ENDPOINT: {AZURE_OPENAI_ENDPOINT}")

credential = AzureCliCredential()

project = AIProjectClient(
    endpoint=PROJECT_ENDPOINT,
    credential=credential,
    allow_preview=True,
)

agent = project.agents.create_version(
    agent_name=AGENT_NAME,
    definition=HostedAgentDefinition(
        protocol_versions=[
            ProtocolVersionRecord(protocol=AgentEndpointProtocol.RESPONSES, version="1.0.0")
        ],
        cpu="1",
        memory="2Gi",
        container_configuration=ContainerConfiguration(image=IMAGE_URI),
        environment_variables={
            "AZURE_OPENAI_ENDPOINT": AZURE_OPENAI_ENDPOINT,
            "AZURE_OPENAI_API_VERSION": AZURE_OPENAI_API_VERSION,
            "AZURE_OPENAI_DEPLOYMENT": AZURE_OPENAI_DEPLOYMENT,
            "APIM_SUBSCRIPTION_KEY": APIM_SUBSCRIPTION_KEY,
            "LOG_LEVEL": "INFO",
        },
    ),
 )

print(f"Agent created: {agent.name}, version: {agent.version}, id {agent.id}")

## 3. Test The Hosted Agent (Direct)

**Purpose:** Validate your agent works by calling the Foundry Responses API directly using your Azure CLI credential.

This is your baseline test—if this fails, APIM proxy tests will also fail. Use this to confirm:
- Agent reached "Running" status ✅
- Network connectivity to Foundry is working ✅
- Authentication with your credential works ✅
- Agent payload format is correct ✅

**How it works:**
- Endpoint: `{PROJECT_ENDPOINT}/agents/{AGENT_NAME}/endpoint/protocols/openai/responses?api-version=v1`
- Auth: Bearer token with `https://ai.azure.com/.default` audience (from your Azure CLI login)
- Headers: Includes `Foundry-Features: HostedAgents=V1Preview` (required for preview access)

If this test succeeds, you know the agent itself is working correctly.

In [ ]:
import json
import requests
from azure.core.credentials import AccessToken
from azure.identity import AzureCliCredential

url = build_foundry_responses_url(PROJECT_ENDPOINT, AGENT_NAME)

if "credential" not in globals():
    credential = AzureCliCredential()

# Get a token for Foundry Responses API
token: AccessToken = credential.get_token("https://ai.azure.com/.default")

headers = {
    "Authorization": f"Bearer {token.token}",
    "Content-Type": DEFAULT_CONTENT_TYPE,
    "Foundry-Features": FOUNDRY_FEATURES_HEADER,
}

payload = {
    "input": DIRECT_TEST_INPUT,
    "stream": False,
}

print(f"POST {url}?api-version={API_VERSION}")
print(f"Agent Name: {AGENT_NAME}")
print(f"Payload: {json.dumps(payload, indent=2)}")
print()

response = requests.post(url, headers=headers, json=payload, params={"api-version": API_VERSION})
print(f"Status: {response.status_code}")
print(f"Headers: {dict(response.headers)}")
print(f"Response text (first 500 chars): {response.text[:500]}")
print()

if response.status_code == 200:
    try:
        result = response.json()
        print("Response JSON:")
        print(json.dumps(result, indent=2))
        if "output_text" in result:
            print(f"\nAgent output:\n{result['output_text']}")
    except json.JSONDecodeError as e:
        print(f"Failed to parse JSON: {e}")
        print(f"Raw response: {response.text}")
else:
    print(f"Error: {response.status_code}")
    if response.text:
        try:
            print(json.dumps(response.json(), indent=2))
        except Exception:
            print(response.text)

## 4. Test The Hosted Agent Via APIM

**Purpose:** Route your request through Azure API Management instead of calling Foundry directly.

This validates the production-like path:
- Client → APIM Gateway (subscription key auth) → Foundry Responses API

**Important: Agent-Specific Routing**

Foundry Hosted Agents **require** the agent name in the URL path. There is no generic `/responses` endpoint that accepts `agent_reference` in the request body. Each agent must have its own URL path:

```
POST https://apim-{APIM_SUFFIX}.azure-api.net/hosted-agent-responses/agents/{AGENT_NAME}/endpoint/protocols/openai/responses
```

**What APIM does automatically:**
- Injects managed identity bearer token for Foundry authentication (audience: `https://ai.azure.com`)
- Enforces `Content-Type: application/json` header
- Injects `Foundry-Features: HostedAgents=V1Preview` header for preview access
- Your client only needs the APIM subscription key (`api-key` header)

**Configuration details** are in [hosted-agent-policy.xml](../../hosted-agent-policy.xml)—update there if you need to change defaults.

Compare this test result with Section 3 to ensure both paths (direct and APIM-proxied) produce consistent responses.

### Get deployment outputs

Before testing via APIM, retrieve the outputs from your infrastructure deployment notebook and update the **top configuration cell** (Cell 3):

- `APIM_SUFFIX`: The unique suffix for your APIM instance
- `APIM_SUBSCRIPTION_KEY`: The starter subscription key from APIM outputs
- `PROJECT_ENDPOINT`: The Foundry agents project endpoint (or let it be computed from `APIM_SUFFIX`)

Keeping these values in one place makes reruns and troubleshooting much easier.

In [ ]:
print("Using APIM/Foundry values from the top configuration cell:")
print(f"APIM_SUFFIX: {APIM_SUFFIX}")
print(f"APIM_SUBSCRIPTION_KEY: {APIM_SUBSCRIPTION_KEY[:20]}..." if len(APIM_SUBSCRIPTION_KEY) > 20 else f"APIM_SUBSCRIPTION_KEY: {APIM_SUBSCRIPTION_KEY}")
print(f"PROJECT_ENDPOINT: {PROJECT_ENDPOINT}")
print()
print("If these are placeholders, update Cell 3 before running APIM tests.")

In [ ]:
import json
import requests

url = build_apim_responses_url(APIM_SUFFIX, AGENT_NAME)

headers = {
    "api-key": APIM_SUBSCRIPTION_KEY,
    "Content-Type": DEFAULT_CONTENT_TYPE,
    "Foundry-Features": FOUNDRY_FEATURES_HEADER,
}

payload = {
    "input": DIRECT_TEST_INPUT,
    "stream": False,
}

print(f"POST {url}?api-version={API_VERSION}")
print(f"APIM Suffix: {APIM_SUFFIX}")
print(f"Agent Name (in URL path): {AGENT_NAME}")
print(f"Content-Type: {DEFAULT_CONTENT_TYPE}")
print(f"Headers: api-key={APIM_SUBSCRIPTION_KEY[:20]}..., Content-Type, Foundry-Features")
print(f"Payload: {json.dumps(payload, indent=2)}")
print()

response = requests.post(url, headers=headers, json=payload, params={"api-version": API_VERSION})
print(f"Status: {response.status_code}")
print()

# Extract response from payload
apim_response = None
agent_text = None

if response.status_code == 200:
    try:
        result = response.json()
        apim_response = result

        # Extract agent text from nested structure
        if "output" in result and len(result["output"]) > 0:
            message = result["output"][0]
            if "content" in message and len(message["content"]) > 0:
                agent_text = message["content"][0].get("text", "")

        print("=" * 80)
        print("AGENT RESPONSE (via APIM):")
        print("=" * 80)
        if agent_text:
            print(agent_text)
        else:
            print("Full response:")
            print(json.dumps(result, indent=2))
        print("=" * 80)

    except json.JSONDecodeError as e:
        print(f"Failed to parse JSON: {e}")
        print(f"Raw response: {response.text}")
else:
    print(f"Error: {response.status_code}")
    if response.text:
        try:
            apim_response = response.json()
            print(json.dumps(apim_response, indent=2))
        except Exception:
            print(response.text)

print("\nExtracted response stored in 'apim_response' variable")
print("Agent text stored in 'agent_text' variable")